# Question 5

## Imports and setup

In [1]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from transformers import AutoProcessor, AutoModel, AutoModelForImageClassification
from sklearn.metrics import f1_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm
import numpy as np

# Optimize for available hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
def load_datasets(processor_transform=None, dataset_name="pets"):
    if dataset_name == "pets":
        train_data = torchvision.datasets.OxfordIIITPet(
            root="./data", split="trainval", transform=processor_transform, download=True
        )
        test_data = torchvision.datasets.OxfordIIITPet(
            root="./data", split="test", transform=processor_transform, download=True
        )
        class_names = train_data.classes
    else :
        train_data = torchvision.datasets.FGVCAircraft(
            root="./data", split="trainval", transform=processor_transform, download=True
        )
        test_data = torchvision.datasets.FGVCAircraft(
            root="./data", split="test", transform=processor_transform, download=True
        )
        class_names = train_data.classes
        
    return train_data, test_data, class_names

## Part A - CLIP Zero-Shot Classification

In [3]:
def get_clip_transform(processor):
    def transform(image):
        processed = processor(images=image, return_tensors="pt")
        return processed['pixel_values'].squeeze(0)
    return transform

In [4]:
def evaluate_zero_shot(model_name, dataset_name, prompt_template):
    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    _, test_data, class_names = load_datasets(get_clip_transform(processor), dataset_name)
    test_loader = DataLoader(test_data, batch_size=16, shuffle=False)
    
    clean_names = [name.replace("_", " ").replace("-", " ") for name in class_names]
    text_inputs = [prompt_template.format(c=name) for name in clean_names]
    
    text_tokens = processor(
        text=text_inputs,
        padding="max_length",
        max_length=64,       
        truncation=True,    
        return_tensors="pt" 
    ).to(device)

    all_preds = []
    all_labels = []

    with torch.inference_mode():
        
        text_outputs = model.get_text_features(**text_tokens)
        text_embeddings = text_outputs.pooler_output
        text_embeddings = text_embeddings / text_embeddings.norm(p=2, dim=-1, keepdim=True)

        for images, labels in tqdm(test_loader, desc=f"Zero-Shot {model_name} , Prompt {prompt_template}"):
            
            images = images.to(device) 
            image_outputs = model.get_image_features(pixel_values=images)    
            image_embeddings = image_outputs.pooler_output
            image_embeddings = image_embeddings / image_embeddings.norm(p=2, dim=-1, keepdim=True)
            
            similarity = image_embeddings @ text_embeddings.T
            preds = similarity.argmax(dim=-1)        
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())            
            
    # Calculate Metrics
    accuracy = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    f1_weighted = f1_score(all_labels, all_preds, average='weighted')

    print("=== Evaluation Results ===")
    print(f"Accuracy:            {accuracy * 100:.2f}%")
    print(f"F1 Score (Macro):    {f1_macro:.4f}")
    print(f"F1 Score (Weighted): {f1_weighted:.4f}")
    print("==========================\n")


In [5]:
dataset_name = "pets"
_, _, class_names = load_datasets(dataset_name=dataset_name)
templates = ["A photo of a {c}.", "Animal with class {c}."]
for prompt in templates:
    evaluate_zero_shot("openai/clip-vit-base-patch32", dataset_name, prompt)

100%|██████████| 792M/792M [00:29<00:00, 27.2MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 11.8MB/s]


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Zero-Shot openai/clip-vit-base-patch32 , Prompt A photo of a {c}.: 100%|██████████| 230/230 [00:41<00:00,  5.55it/s]


=== Evaluation Results ===
Accuracy:            85.15%
F1 Score (Macro):    0.8381
F1 Score (Weighted): 0.8412



Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Zero-Shot openai/clip-vit-base-patch32 , Prompt Animal with class {c}.: 100%|██████████| 230/230 [00:41<00:00,  5.56it/s]

=== Evaluation Results ===
Accuracy:            84.33%
F1 Score (Macro):    0.8286
F1 Score (Weighted): 0.8315



In [6]:
dataset_name = "aircraft"
_, _, class_names = load_datasets(dataset_name=dataset_name)
templates = ["A photo of a {c}.", "A photo of a {c}, a type of aircraft."]
for prompt in templates:
    evaluate_zero_shot("openai/clip-vit-base-patch32", dataset_name, prompt)

100%|██████████| 2.75G/2.75G [01:13<00:00, 37.2MB/s]


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Zero-Shot openai/clip-vit-base-patch32 , Prompt A photo of a {c}.: 100%|██████████| 209/209 [01:11<00:00,  2.92it/s]


=== Evaluation Results ===
Accuracy:            17.88%
F1 Score (Macro):    0.1550
F1 Score (Weighted): 0.1551



Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Zero-Shot openai/clip-vit-base-patch32 , Prompt A photo of a {c}, a type of aircraft.: 100%|██████████| 209/209 [01:10<00:00,  2.97it/s]

=== Evaluation Results ===
Accuracy:            18.57%
F1 Score (Macro):    0.1690
F1 Score (Weighted): 0.1691



## Part B - SigLIP Zero-Shot Classification

In [7]:
dataset_name = "pets"
_, _, class_names = load_datasets(dataset_name=dataset_name)
templates = ["A photo of a {c}.", "Animal with class {c}."]
for prompt in templates:
    evaluate_zero_shot("google/siglip2-base-patch16-224", dataset_name, prompt)

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/253 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

...

Zero-Shot google/siglip2-base-patch16-224 , Prompt A photo of a {c}.: 100%|██████████| 230/230 [01:07<00:00,  3.40it/s]


=== Evaluation Results ===
Accuracy:            10.06%
F1 Score (Macro):    0.0362
F1 Score (Weighted): 0.0361



Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Zero-Shot google/siglip2-base-patch16-224 , Prompt Animal with class {c}.: 100%|██████████| 230/230 [01:08<00:00,  3.35it/s]


=== Evaluation Results ===
Accuracy:            5.70%
F1 Score (Macro):    0.0094
F1 Score (Weighted): 0.0094



In [8]:
dataset_name = "aircraft"
_, _, class_names = load_datasets(dataset_name=dataset_name)
templates = ["A photo of a {c}.", "A photo of a {c}, a type of aircraft."]
for prompt in templates:
    evaluate_zero_shot("google/siglip2-base-patch16-224", dataset_name, prompt)

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Zero-Shot google/siglip2-base-patch16-224 , Prompt A photo of a {c}.: 100%|██████████| 209/209 [01:34<00:00,  2.22it/s]


=== Evaluation Results ===
Accuracy:            11.70%
F1 Score (Macro):    0.0623
F1 Score (Weighted): 0.0624



Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Zero-Shot google/siglip2-base-patch16-224 , Prompt A photo of a {c}, a type of aircraft.: 100%|██████████| 209/209 [01:38<00:00,  2.12it/s]


=== Evaluation Results ===
Accuracy:            17.73%
F1 Score (Macro):    0.1063
F1 Score (Weighted): 0.1063



## Part C - CLIP, SigLIP linear probe

In [9]:
def extract_features(model, dataloader, device, model_type="hf_clip_siglip"):
    model.eval()
    features = []
    labels_list = []
    
    with torch.inference_mode():
        for images, labels in tqdm(dataloader, desc=f"Extracting Features ({model_type})"):
            images = images.to(device)
            
            if model_type == "hf_clip_siglip":
                outputs = model.get_image_features(pixel_values=images)
                if hasattr(outputs, 'pooler_output'):
                    emb = outputs.pooler_output
                else:
                    emb = outputs
            
            elif model_type == "hf_dino":
                outputs = model(pixel_values=images)
                emb = outputs.last_hidden_state[:, 0, :] 
                
            elif model_type == "torchvision":
                emb = model(images)
            
            emb = emb / emb.norm(p=2, dim=-1, keepdim=True)
            
            features.append(emb.cpu().numpy())
            labels_list.append(labels.numpy())
    return np.vstack(features), np.concatenate(labels_list)

In [10]:
def run_linear_probe(train_features, train_labels, test_features, test_labels, class_names):
    
    print("Training Linear Classifier...")
    clf = LogisticRegression(max_iter=1000, n_jobs=-1)
    clf.fit(train_features, train_labels)
    
    print("Evaluating...")
    preds = clf.predict(test_features)
    
    accuracy = accuracy_score(test_labels, preds)
    f1_macro = f1_score(test_labels, preds, average='macro')
    
    print("=== Linear Probe Results ===")
    print(f"Accuracy:         {accuracy * 100:.2f}%")
    print(f"F1 Score (Macro): {f1_macro:.4f}")
    print("============================\n")
    
    return clf 

In [11]:
import gc 

datasets = ["pets", "aircraft"]
models = ["openai/clip-vit-base-patch32", "google/siglip2-base-patch16-224"]

for dataset_name in datasets: 
    for model_name in models:
        print(f"\n========== Evaluating {model_name} on {dataset_name} ==========")
        
        processor = AutoProcessor.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name).to(device)
        model.eval()

        train_data, test_data, class_names = load_datasets(get_clip_transform(processor), dataset_name)
        
        train_loader = DataLoader(train_data, batch_size=64, shuffle=False)
        test_loader = DataLoader(test_data, batch_size=64, shuffle=False)
    
        train_feats, train_labels = extract_features(model, train_loader, device, "hf_clip_siglip")
        test_feats, test_labels = extract_features(model, test_loader, device, "hf_clip_siglip")
        
        run_linear_probe(train_feats, train_labels, test_feats, test_labels, class_names)
        
        del model
        del processor
        gc.collect()
        torch.cuda.empty_cache()


========== Evaluating openai/clip-vit-base-patch32 on pets ==========


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Extracting Features (hf_clip_siglip): 100%|██████████| 58/58 [00:40<00:00,  1.44it/s]

Training Linear Classifier...


Evaluating...
=== Linear Probe Results ===
Accuracy:         85.50%
F1 Score (Macro): 0.8532


========== Evaluating google/siglip2-base-patch16-224 on pets ==========


...

Evaluating...
=== Linear Probe Results ===
Accuracy:         38.40%
F1 Score (Macro): 0.3537


========== Evaluating google/siglip2-base-patch16-224 on aircraft ==========


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Extracting Features (hf_clip_siglip): 100%|██████████| 53/53 [01:40<00:00,  1.90s/it]

Training Linear Classifier...


Evaluating...
=== Linear Probe Results ===
Accuracy:         75.25%
F1 Score (Macro): 0.7460



## Part D - DINOV1, DINOV2 linear probe 

In [12]:
import gc 

datasets = ["pets", "aircraft"]
models = ["facebook/dino-vitb16", "facebook/dinov2-base"]

for dataset_name in datasets: 
    for model_name in models:
        print(f"\n========== Evaluating {model_name} on {dataset_name} ==========")
        
        processor = AutoProcessor.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name).to(device)
        model.eval()

        train_data, test_data, class_names = load_datasets(get_clip_transform(processor), dataset_name)
        
        train_loader = DataLoader(train_data, batch_size=64, shuffle=False)
        test_loader = DataLoader(test_data, batch_size=64, shuffle=False)
    
        train_feats, train_labels = extract_features(model, train_loader, device, "hf_dino")
        test_feats, test_labels = extract_features(model, test_loader, device, "hf_dino")
        
        run_linear_probe(train_feats, train_labels, test_feats, test_labels, class_names)
        
        del model
        del processor
        gc.collect()
        torch.cuda.empty_cache()


========== Evaluating facebook/dino-vitb16 on pets ==========


preprocessor_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


pytorch_model.bin:   0%|          | 0.00/343M [00:00<?, ?B/s]

...

Evaluating...
=== Linear Probe Results ===
Accuracy:         43.35%
F1 Score (Macro): 0.4098


========== Evaluating facebook/dinov2-base on aircraft ==========


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Extracting Features (hf_dino): 100%|██████████| 53/53 [01:44<00:00,  1.98s/it]

Training Linear Classifier...


Evaluating...
=== Linear Probe Results ===
Accuracy:         55.78%
F1 Score (Macro): 0.5500



## Part E - VitBase ImageNet

In [13]:
import gc 
from torchvision.models import vit_b_16, ViT_B_16_Weights
import torch.nn as nn

datasets = ["pets", "aircraft"]

for dataset_name in datasets: 
    print(f"\n========== Evaluating VitBase ImageNet on {dataset_name} ==========")

    weights = ViT_B_16_Weights.IMAGENET1K_V1
    vit_sup = vit_b_16(weights=weights).to(device)

    vit_sup.heads = nn.Identity()
    vit_transform = weights.transforms()

    train_data, test_data, class_names = load_datasets(vit_transform, dataset_name)
    
    train_loader = DataLoader(train_data, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

    train_feats, train_labels = extract_features(vit_sup, train_loader, device, "torchvision")
    test_feats, test_labels = extract_features(vit_sup, test_loader, device, "torchvision")
    
    run_linear_probe(train_feats, train_labels, test_feats, test_labels, class_names)
    
    del vit_sup
    gc.collect()
    torch.cuda.empty_cache()


========== Evaluating VitBase ImageNet on pets ==========
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 191MB/s]
Extracting Features (torchvision): 100%|██████████| 58/58 [01:03<00:00,  1.10s/it]


Training Linear Classifier...
Evaluating...
=== Linear Probe Results ===
Accuracy:         92.72%
F1 Score (Macro): 0.9266


========== Evaluating VitBase ImageNet on aircraft ==========


Extracting Features (torchvision): 100%|██████████| 53/53 [01:30<00:00,  1.71s/it]

Training Linear Classifier...


Evaluating...
=== Linear Probe Results ===
Accuracy:         36.87%
F1 Score (Macro): 0.3456

